# A/B Test Result Analysis

## Project Overview
Analyze experiment data to compare control vs treatment on conversion and supporting metrics, including statistical significance, confidence intervals, and practical significance.

## Learning Objectives
- Design and interpret A/B test results with proper statistical rigor
- Calculate conversion rate differences with confidence intervals
- Perform chi-squared and z-proportion tests for significance
- Assess practical significance vs statistical significance
- Analyze segment-level treatment effects
- Understand common pitfalls in A/B testing

## Problem Statement
A product team ran a website redesign experiment (control = old design, treatment = new design) over 4 weeks. We need to determine whether the treatment improved conversion rate and other engagement metrics — and whether the improvement is practically meaningful.

## Why This Project Matters
A/B testing is the gold standard for causal inference in tech companies. Misinterpreting results leads to shipping harmful changes or killing beneficial ones. Understanding uncertainty, sample size, and effect size is essential for data-driven product decisions.

## Dataset Overview
Synthetic A/B test data: ~20,000 visitors split between control and treatment groups, with conversion, revenue, pages viewed, and device/segment metadata.

## Dataset Source and License Notes
- Synthetic data generated for educational purposes
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
ALPHA = 0.05  # significance level
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 20000
group = np.random.choice(['control', 'treatment'], size=n, p=[0.5, 0.5])
device = np.random.choice(['desktop', 'mobile', 'tablet'], size=n, p=[0.45, 0.45, 0.10])
segment = np.random.choice(['new', 'returning'], size=n, p=[0.6, 0.4])
day = np.random.randint(1, 29, size=n)  # 4-week experiment

# Base conversion rates
base_cvr = {'control': 0.12, 'treatment': 0.135}  # ~12.5% relative lift
# Device modifiers
device_mod = {'desktop': 1.0, 'mobile': 0.85, 'tablet': 0.90}
# Segment modifiers
seg_mod = {'new': 0.80, 'returning': 1.30}

conv_probs = np.array([base_cvr[g] * device_mod[d] * seg_mod[s]
                        for g, d, s in zip(group, device, segment)])
conv_probs = np.clip(conv_probs, 0, 0.5)
converted = np.random.binomial(1, conv_probs)

# Revenue (only for converters)
revenue = np.where(converted == 1, np.random.lognormal(3.5, 0.8, n).round(2), 0)

# Pages viewed
pages = np.random.poisson(
    np.where(group == 'treatment', 4.2, 3.8) *
    np.where(device == 'mobile', 0.7, 1.0), n)
pages = np.clip(pages, 1, 30)

df = pd.DataFrame({
    'visitor_id': np.arange(n),
    'group': group,
    'device': device,
    'segment': segment,
    'day': day,
    'converted': converted,
    'revenue': revenue,
    'pages_viewed': pages
})
print(f'Dataset: {df.shape}')
print(f'\nGroup distribution:\n{df["group"].value_counts()}')
print(f'\nOverall conversion rate: {df["converted"].mean():.4f}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nGroup sizes: {df["group"].value_counts().to_dict()}')
print(f'Device distribution:\n{df["device"].value_counts()}')
print(f'\nSegment distribution:\n{df["segment"].value_counts()}')
# Check randomization balance
for col in ['device', 'segment']:
    ct = pd.crosstab(df['group'], df[col], normalize='index')
    print(f'\n{col} balance (row %):\n{ct.round(3)}')

## Exploratory Data Analysis
### Conversion Rate by Group

In [ ]:
cvr = df.groupby('group')['converted'].agg(['mean', 'sum', 'count'])
cvr.columns = ['conversion_rate', 'conversions', 'visitors']
print(cvr)
print(f'\nAbsolute lift: {cvr.loc["treatment", "conversion_rate"] - cvr.loc["control", "conversion_rate"]:.4f}')
print(f'Relative lift: {(cvr.loc["treatment", "conversion_rate"] / cvr.loc["control", "conversion_rate"] - 1) * 100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# Bar chart
cvr['conversion_rate'].plot.bar(ax=axes[0], color=['#2196F3', '#FF9800'], edgecolor='black')
axes[0].set_title('Conversion Rate by Group')
axes[0].set_ylabel('Conversion Rate')
axes[0].tick_params(axis='x', rotation=0)

# Revenue per visitor
rpv = df.groupby('group')['revenue'].mean()
rpv.plot.bar(ax=axes[1], color=['#2196F3', '#FF9800'], edgecolor='black')
axes[1].set_title('Revenue per Visitor by Group')
axes[1].set_ylabel('Avg Revenue ($)')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'conversion_comparison.png'), dpi=100, bbox_inches='tight')
plt.show()

## Statistical Significance Test

In [ ]:
# Z-test for proportions
ctrl = df[df['group'] == 'control']
treat = df[df['group'] == 'treatment']

n_c, n_t = len(ctrl), len(treat)
p_c = ctrl['converted'].mean()
p_t = treat['converted'].mean()

# Pooled proportion
p_pool = (ctrl['converted'].sum() + treat['converted'].sum()) / (n_c + n_t)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_c + 1/n_t))
z_stat = (p_t - p_c) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print(f'Control CVR:   {p_c:.4f}')
print(f'Treatment CVR: {p_t:.4f}')
print(f'Difference:    {p_t - p_c:.4f}')
print(f'Z-statistic:   {z_stat:.4f}')
print(f'P-value:       {p_value:.6f}')
print(f'Significant at α={ALPHA}? {"YES" if p_value < ALPHA else "NO"}')

## Confidence Interval for the Difference

In [ ]:
# 95% CI for the difference in proportions
se_diff = np.sqrt(p_c * (1 - p_c) / n_c + p_t * (1 - p_t) / n_t)
z_crit = stats.norm.ppf(1 - ALPHA / 2)
diff = p_t - p_c
ci_low = diff - z_crit * se_diff
ci_high = diff + z_crit * se_diff

print(f'Difference: {diff:.4f}')
print(f'95% CI: [{ci_low:.4f}, {ci_high:.4f}]')
print(f'Entire CI above 0? {"YES — treatment wins" if ci_low > 0 else "NO — includes zero"}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(0, diff, xerr=z_crit * se_diff, color='#FF9800', edgecolor='black', capsize=8, height=0.4)
ax.axvline(0, color='red', linestyle='--', label='No effect')
ax.set_yticks([0])
ax.set_yticklabels(['Treatment - Control'])
ax.set_xlabel('Conversion Rate Difference')
ax.set_title('95% Confidence Interval for CVR Lift')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confidence_interval.png'), dpi=100, bbox_inches='tight')
plt.show()

## Practical Significance

In [ ]:
# Minimum detectable effect (MDE) — business defines this
MDE = 0.01  # 1 percentage point lift considered practically meaningful
print(f'Observed lift: {diff:.4f}')
print(f'MDE threshold: {MDE}')
print(f'Practically significant? {"YES" if diff >= MDE else "NO"}')
print(f'\nEven if statistically significant, ask: is {diff*100:.1f} pp lift worth the engineering cost?')

## Segment-Level Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By device
device_cvr = df.groupby(['group', 'device'])['converted'].mean().unstack(level=0)
device_cvr.plot.bar(ax=axes[0], edgecolor='black')
axes[0].set_title('CVR by Device × Group')
axes[0].set_ylabel('Conversion Rate')
axes[0].tick_params(axis='x', rotation=0)

# By segment
seg_cvr = df.groupby(['group', 'segment'])['converted'].mean().unstack(level=0)
seg_cvr.plot.bar(ax=axes[1], edgecolor='black')
axes[1].set_title('CVR by Segment × Group')
axes[1].set_ylabel('Conversion Rate')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'segment_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

# Print lift by segment
for col in ['device', 'segment']:
    print(f'\nLift by {col}:')
    for val in df[col].unique():
        sub = df[df[col] == val]
        c = sub[sub['group'] == 'control']['converted'].mean()
        t = sub[sub['group'] == 'treatment']['converted'].mean()
        print(f'  {val}: {c:.4f} → {t:.4f} (lift: {(t/c-1)*100:.1f}%)')

## Daily Conversion Trend

In [ ]:
daily = df.groupby(['day', 'group'])['converted'].mean().unstack()
fig, ax = plt.subplots(figsize=(14, 5))
daily.plot(ax=ax, marker='o', markersize=3)
ax.set_title('Daily Conversion Rate by Group')
ax.set_xlabel('Experiment Day')
ax.set_ylabel('Conversion Rate')
ax.legend(title='Group')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'daily_trend.png'), dpi=100, bbox_inches='tight')
plt.show()

## Pages Viewed Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for g, color in [('control', '#2196F3'), ('treatment', '#FF9800')]:
    sub = df[df['group'] == g]['pages_viewed']
    ax.hist(sub, bins=range(1, 20), alpha=0.5, color=color, label=g, edgecolor='black')
ax.set_title('Pages Viewed Distribution by Group')
ax.set_xlabel('Pages Viewed')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'pages_viewed.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'Avg pages: control={ctrl["pages_viewed"].mean():.2f}, treatment={treat["pages_viewed"].mean():.2f}')
t_stat, p_pages = stats.ttest_ind(ctrl['pages_viewed'], treat['pages_viewed'])
print(f'T-test p-value: {p_pages:.6f}')

## Interpretation and Business Insight
- **Treatment increased conversion rate** by ~1.5 percentage points (relative lift ~12%)
- The result is **statistically significant** at the 5% level — unlikely due to chance
- The **confidence interval** excludes zero, confirming a real positive effect
- The lift is **consistent across devices and segments** — no Simpson's paradox
- **Returning users** show higher absolute conversion in both groups — the treatment helps acquisition and retention
- **Pages viewed** also increased in treatment — the redesign improves engagement beyond conversion
- **Practical significance**: a 1.5 pp lift on 20K+ visitors/month = meaningful revenue impact

## Limitations
- Synthetic data — real experiments have network effects, novelty bias, and learning effects
- No time-based novelty/fatigue analysis (would need longer experiment)
- Revenue is modeled independently of conversion behavior
- No multi-armed bandit or sequential testing comparison
- No power analysis for sample size planning

## How to Improve This Project
- Add Bayesian A/B testing for probability-of-improvement estimates
- Include sequential testing / early stopping analysis
- Add power analysis and sample size calculation
- Include multi-metric correction (Bonferroni / FDR)
- Analyze time-to-convert, not just binary conversion

## Production Considerations
- Implement guardrail metrics (latency, error rate) alongside primary metrics
- Use sequential testing frameworks for early stopping
- Ensure proper randomization unit (user-level, not session-level)
- Track post-experiment holdback groups for long-term effects
- Automate experiment result reporting with confidence intervals

## Common Mistakes
- Peeking at results before the experiment reaches planned sample size
- Ignoring practical significance — celebrating a statistically significant but tiny lift
- Not checking for Sample Ratio Mismatch (SRM) before analyzing results
- Running multiple tests without correcting for multiple comparisons
- Confusing per-session and per-user conversion rates

## Mini Challenge / Exercises
1. Calculate the required sample size to detect a 1 pp lift with 80% power at α=0.05.
2. Run a chi-squared test instead of the z-test — do the results match?
3. What happens to the confidence interval if you use only the first 7 days of data?
4. Compute the Bayesian probability that treatment is better than control using Beta distributions.

## Final Summary / Key Takeaways
- A/B testing requires both statistical and practical significance assessment
- Confidence intervals are more informative than p-values alone
- Segment analysis ensures the treatment doesn't help one group at another's expense
- Randomization checks (SRM) should happen before any analysis
- The treatment shows a meaningful, consistent lift — recommendation: ship it